<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_extension_array.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 扩展数组（Extension Arrays）¶

Lance 为 **Arrow 数组** 和 **Pandas Series** 提供了扩展支持，用于表示机器学习相关的数据类型。

---

### **BFloat16（半精度浮点数）¶**

**BFloat16** 是一种专为机器学习设计的 **16 位浮点数格式**。
直观理解，它只有约 **2–3 位有效精度**，但其表示范围与 32 位浮点数相同（约 `1e-38` 到 `1e38`）。
相比之下，普通的 16 位浮点数（float16）的范围约为 `5.96e-8` 到 `65504`。

Lance 提供了：

* Arrow 扩展数组：`lance.arrow.BFloat16Array`
* Pandas 扩展数组：`lance._arrow.PandasBFloat16Type`

这两者都与 **`ml_dtypes`** 中的 **bfloat16 NumPy 扩展类型** 兼容。

---

#### ✅ 在 Pandas 中使用

```python
import lance.arrow
import pandas as pd

pd.Series([1.1, 2.1, 3.4], dtype="lance.bfloat16")
# 输出：
# 0    1.1015625
# 1      2.09375
# 2      3.40625
# dtype: lance.bfloat16
```

---

#### ✅ 创建 Arrow 数组

```python
from lance.arrow import bfloat16_array

bfloat16_array([1.1, 2.1, 3.4])
# <lance.arrow.BFloat16Array object at 0x...>
# [
#   1.1015625,
#   2.09375,
#   3.40625
# ]
```

---

#### ✅ 从 NumPy 数组转换

```python
import numpy as np
from ml_dtypes import bfloat16
from lance.arrow import PandasBFloat16Array, BFloat16Array

np_array = np.array([1.1, 2.1, 3.4], dtype=bfloat16)

PandasBFloat16Array.from_numpy(np_array)
# <PandasBFloat16Array>
# [1.1015625, 2.09375, 3.40625]
# Length: 3, dtype: lance.bfloat16

BFloat16Array.from_numpy(np_array)
# <lance.arrow.BFloat16Array object at 0x...>
# [
#   1.1015625,
#   2.09375,
#   3.40625
# ]
```

读取时，可以通过各数组类的 `to_numpy()` 方法将其转换回 NumPy 的 `bfloat16` 类型。

---

### **ImageURI（图像 URI 数组）¶**

`lance.arrow.ImageURIArray` 是一种用于**存储图像在外部存储系统中路径**的数组类型。

例如：

* 本地文件：`file:///path/to/image.png`
* AWS S3：`s3://bucket/path/image.jpeg`

当你想从已有存储中**延迟加载图像**时，可以使用该类型。

---

#### ✅ 创建 ImageURIArray

```python
from lance.arrow import ImageURIArray

ImageURIArray.from_uris([
   "/tmp/image1.jpg",
   "file:///tmp/image2.jpg",
   "s3://example/image3.jpg"
])
# <lance.arrow.ImageURIArray object at 0x...>
# ['/tmp/image1.jpg', 'file:///tmp/image2.jpg', 's3://example/image3.jpg']
```

URI 不会被严格验证，图像也不会自动读取进内存。

---

#### ✅ 读取图像内容

```python
from lance.arrow import ImageURIArray
import os

relative_path = "images/1.png"
uris = [os.path.join(os.path.dirname(__file__), relative_path)]
ImageURIArray.from_uris(uris).read_uris()
# <lance.arrow.EncodedImageArray object at 0x...>
# [b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00...']
```

---

### **EncodedImage（编码图像数组）¶**

`lance.arrow.EncodedImageArray` 存储 **JPEG/PNG 等编码压缩后的图像数据**。
用于从磁盘读取或在 HTML 中嵌入图像等场景。

可通过以下方式创建：

* 调用 `ImageURIArray.read_uris()` 读取图像；
* 调用 `ImageArray.from_array()` 并传入 `pyarrow.BinaryArray`；
* 调用 `ImageTensorArray.to_encoded()`。

---

#### ✅ 解码为张量

`EncodedImageArray.to_tensor()` 方法可将压缩图像解码为 `lance.arrow.FixedShapeImageTensorArray`。
解码顺序优先使用：

1. 自定义 `decoder` 参数；
2. `Pillow`；
3. `TensorFlow`。
   若都不可用，将抛出异常。

```python
from lance.arrow import ImageURIArray
import os

uris = [os.path.join(os.path.dirname(__file__), "images/1.png")]
encoded_images = ImageURIArray.from_uris(uris).read_uris()
print(encoded_images.to_tensor())

# 使用 TensorFlow 自定义解码器
def tensorflow_decoder(images):
    import tensorflow as tf
    import numpy as np
    return np.stack(tf.io.decode_png(img.as_py(), channels=3) for img in images.storage)

print(encoded_images.to_tensor(tensorflow_decoder))
```

输出示例：

```
<lance.arrow.FixedShapeImageTensorArray object at 0x...>
[[42, 42, 42, 255]]
```

---

### **FixedShapeImageTensor（固定形状图像张量）¶**

`lance.arrow.FixedShapeImageTensorArray` 用于存储图像的**张量表示**，
每个像素点为数值形式，通常为形状 `(height, width, channels)` 的三维张量。

颜色图像中，每个像素对应三个通道值（RGB）。

这些图像可以单独转换为 NumPy 数组，
也可以堆叠为形状为 `(batch_size, height, width, channels)` 的四维数组。

---

#### ✅ 创建方式

* 通过 `EncodedImageArray.to_tensor()` 解码得到；
* 或通过 `ImageArray.from_array()` 并传入 `pyarrow.FixedShapeTensorArray` 创建。

---

#### ✅ 编码回图像

可通过 `to_encoded()` 方法编码为 `EncodedImageArray`。
如果未提供自定义编码器，将按以下顺序尝试：

1. TensorFlow
2. Pillow

默认编码格式为 **PNG**，若均不可用会抛出异常。

```python
from lance.arrow import ImageURIArray

uris = [image_uri]
tensor_images = ImageURIArray.from_uris(uris).read_uris().to_tensor()
tensor_images.to_encoded()
# <lance.arrow.EncodedImageArray object at 0x...>
# [... b'\x89PNG\r\n\x1a...']
```
